# Salient Object Detection — Demo Notebook

Upload an image (or pick one from your test split) and visualise:

1. Input image
2. Predicted saliency mask
3. Overlay (input + predicted mask)
4. Inference time per image


In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from sod_model import build_model

## 1. Load a trained checkpoint

In [ ]:
WEIGHTS = 'checkpoints/best_improved.pt'
IMG_SIZE = 128

device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

ckpt = torch.load(WEIGHTS, map_location=device)
model_name = ckpt.get('config', {}).get('model', 'improved')
model = build_model(model_name).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded {WEIGHTS}  model={model_name}  device={device}')

: 

## 2. Pick an image

In [ ]:
IMAGE_PATH = 'data/duts/images/test/ILSVRC2012_test_00000003.jpg'
pil_img = Image.open(IMAGE_PATH).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
arr = np.array(pil_img, dtype=np.float32) / 255.0
x = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(device)
print('input tensor shape:', x.shape)

## 3. Run inference

In [ ]:
with torch.no_grad():
    t0 = time.time()
    prob = torch.sigmoid(model(x))[0, 0].cpu().numpy()
    dt_ms = (time.time() - t0) * 1000.0
binary = (prob > 0.5).astype(np.float32)
print(f'Inference time: {dt_ms:.1f} ms')

## 4. Visualise

In [ ]:
def overlay(img_arr, mask, alpha=0.45):
    color = np.zeros_like(img_arr); color[..., 0] = 1.0
    m = mask[..., None]
    return np.clip(img_arr * (1 - alpha * m) + color * (alpha * m), 0, 1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(arr); axes[0].set_title('Input'); axes[0].axis('off')
axes[1].imshow(prob, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Predicted (prob)'); axes[1].axis('off')
axes[2].imshow(binary, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Predicted (binary)'); axes[2].axis('off')
axes[3].imshow(overlay(arr, binary)); axes[3].set_title(f'Overlay  ({dt_ms:.1f} ms)')
axes[3].axis('off')
plt.tight_layout()
plt.show()